In [1]:
import os
# Set CUDA visible devices to avoid memory conflicts
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import sys
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import cv2
from pathlib import Path

# Add project root to path to import modules
project_root = Path(__file__).resolve().parents[3] if '__file__' in globals() else Path('.').resolve().parents[3]
sys.path.append(str(project_root / 'src'))
os.chdir(str(project_root / 'jhpark/image-artifacts/src'))


print(f"Project root: {project_root}")
print(f"Current working directory: {os.getcwd()}")

# Set up visualization
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12


Project root: /home
Current working directory: /home/jhpark/image-artifacts/src


In [2]:
# Load data from a single image directory
def load_metadata(image_dir_path):
    """Load metadata.pkl from an image directory"""
    metadata_path = os.path.join(image_dir_path, 'metadata.pkl')
    if not os.path.exists(metadata_path):
        raise FileNotFoundError(f"metadata.pkl not found in {image_dir_path}")
    
    with open(metadata_path, 'rb') as f:
        data = pickle.load(f)
    return data

def examine_data_structure(data):
    """Examine and print the structure of loaded data"""
    print("=== DATA STRUCTURE OVERVIEW ===")
    print(f"Top-level keys: {list(data.keys())}")
    print()
    
    # ID
    if 'id' in data:
        print(f"🆔 ID: {data['id']}")
        print()
    
    # Real image path
    if 'real_image_path' in data:
        print(f"📁 REAL IMAGE PATH: {data['real_image_path']}")
        # Try to get image dimensions from the actual file
        try:
            import PIL.Image
            img = PIL.Image.open(data['real_image_path'])
            print(f"📐 IMAGE DIMENSIONS: {img.width} x {img.height}")
            img.close()
        except Exception as e:
            print(f"⚠️  Could not load image: {e}")
        print()
    
    # Caption
    if 'caption' in data:
        caption = data['caption']
        print(f"💬 CAPTION: \"{caption}\"")
        print()
    
    # Artifacts
    if 'artifacts' in data:
        print("🎯 ARTIFACTS:")
        artifacts = data['artifacts']
        print(f"  📊 Total artifacts: {len(artifacts)}")
        print()
        
        for i, artifact in enumerate(artifacts):
            print(f"  📌 ARTIFACT #{i+1}:")
            artifact_type = artifact.get('artifact_type', 'unknown')
            print(f"    🏷️  Type: {artifact_type}")
            print(f"    🎭 Entity: {artifact.get('entity', 'N/A')}")
            print(f"    🎯 Subentity: {artifact.get('subentity', 'N/A')}")
            
            # Bounding boxes
            if 'target_bbox' in artifact:
                bbox = artifact['target_bbox']
                print(f"    📦 Target BBox: {bbox}")
            if 'reference_bbox' in artifact:
                bbox = artifact['reference_bbox']
                print(f"    📦 Reference BBox: {bbox}")
            
            # Patch indices
            if 'target_patch_indices' in artifact:
                patches = artifact['target_patch_indices']
                print(f"    🎯 Target patches: {len(patches)} (sample: {patches[:5]}...)")
            if 'reference_patch_indices' in artifact:
                patches = artifact['reference_patch_indices']
                print(f"    🎯 Reference patches: {len(patches)} (sample: {patches[:5]}...)")
            
            # Masks
            if 'target_mask' in artifact:
                mask = artifact['target_mask']
                print(f"    🎭 Target mask shape: {mask.shape}")
            if 'reference_mask' in artifact:
                mask = artifact['reference_mask']
                print(f"    🎭 Reference mask shape: {mask.shape}")
            
            # Distortion kernel (for distortion artifacts)
            if 'distortion_kernel' in artifact:
                kernel = artifact['distortion_kernel']
                print(f"    🌀 Distortion kernel: {kernel}")
            
            print()



In [3]:
# Visualize the input data and pre-generated images
def visualize_input_data(selected_dir, metadata, width_ratios=(1, 2)):
    """
    Visualize the input data and existing generated images
    
    Args:
        selected_dir: Directory containing the image data
        metadata: Metadata dictionary
        width_ratios: Tuple specifying relative widths of the two plots (default: (1, 1))
    """
    
    # Load the pre-generated images
    detection_img_path = selected_dir / "02_detection_results.png"
    patch_masks_path = selected_dir / "03_patch_masks_removal.png"
    
    # Create figure with custom width ratios
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    gs = fig.add_gridspec(1, 2, width_ratios=width_ratios)
    
    # Clear existing axes and create new ones with gridspec
    for ax in axes:
        ax.remove()
    
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    
    # 1. Detection Results
    if detection_img_path.exists():
        img = Image.open(detection_img_path)
        ax1.imshow(img)
        ax1.set_title("Detection Results", fontsize=14, fontweight='bold')
        ax1.axis('off')
    
    # 2. Patch Masks
    if patch_masks_path.exists():
        img = Image.open(patch_masks_path)
        ax2.imshow(img)
        ax2.set_title("Patch Masks (Removal)", fontsize=14, fontweight='bold')
        ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print some statistics
    print("\\n=== IMAGE INFORMATION ===")
    
    # Get artifact data for this specific case
    if 'artifacts' in metadata:
        artifact_type = list(metadata['artifacts'].keys())[0]  # Should be 'removal'
        artifact_data = metadata['artifacts'][artifact_type]
        
        print(f"\\nArtifact type: {artifact_type}")
        print(f"Target class: {artifact_data.get('class_name', 'N/A')}")
        
        if 'annotation' in artifact_data and 'bounding_box_ref' in artifact_data['annotation']:
            bbox = artifact_data['annotation']['bounding_box_ref']
            width = bbox['xmax'] - bbox['xmin']
            height = bbox['ymax'] - bbox['ymin']
            print(f"Reference bbox size: {width} x {height} pixels")
        
        if 'patch_data' in artifact_data:
            patch_data = artifact_data['patch_data']
            ref_patches = len(patch_data.get('reference_patch_indices', []) or [])
            target_patches = len(patch_data.get('target_patch_indices', []) or [])
            print(f"Patches used - Reference: {ref_patches}, Target: {target_patches}")


In [4]:
# Analyze patch system and visualize patch indices
def visualize_patch_system(metadata):
    """Visualize how the patch system works for all artifacts
    
    Args:
        metadata: Metadata dictionary with new structure
    """
    
    if 'artifacts' not in metadata:
        print("No artifacts found in metadata")
        return
        
    artifacts = metadata['artifacts']
    if not artifacts:
        print("Artifacts list is empty")
        return
    
    # Load image from real_image_path
    if 'real_image_path' not in metadata:
        print("No real_image_path found in metadata")
        return
    
    try:
        import PIL.Image
        img = PIL.Image.open(metadata['real_image_path'])
        img_array = np.array(img)
        img.close()
    except Exception as e:
        print(f"Could not load image: {e}")
        return
    
    print("=== PATCH SYSTEM ANALYSIS (ALL ARTIFACTS) ===")
    print(f"Total artifacts: {len(artifacts)}")
    print(f"Image shape: {img_array.shape}")
    
    # Use standard patch size of 16 for FLUX
    patch_size = 16
    print(f"Patch size: {patch_size}")
    
    # Get patch dimensions
    H, W = img_array.shape[:2]
    patch_H = H // patch_size
    patch_W = W // patch_size
    print(f"Patch grid: {patch_H} x {patch_W} = {patch_H * patch_W} total patches")
    
    # Collect all patch indices from all artifacts
    all_ref_patch_indices = []
    all_target_patch_indices = []
    
    print("\n=== ARTIFACT DETAILS ===")
    for i, artifact in enumerate(artifacts):
        artifact_type = artifact.get('artifact_type', 'unknown')
        entity = artifact.get('entity', 'N/A')
        subentity = artifact.get('subentity', 'N/A')
        
        print(f"Artifact #{i+1}: {artifact_type} - {entity} ({subentity})")
        
        # Get patch indices from this artifact
        ref_indices = artifact.get('reference_patch_indices', []) or []
        target_indices = artifact.get('target_patch_indices', []) or []
        
        print(f"  Reference patches: {len(ref_indices)}")
        print(f"  Target patches: {len(target_indices)}")
        
        # Add to combined lists
        all_ref_patch_indices.extend(ref_indices)
        all_target_patch_indices.extend(target_indices)
    
    # Remove duplicates while preserving order
    all_ref_patch_indices = list(dict.fromkeys(all_ref_patch_indices))
    all_target_patch_indices = list(dict.fromkeys(all_target_patch_indices))
    
    print(f"\n=== COMBINED PATCHES ===")
    print(f"Total unique reference patches: {len(all_ref_patch_indices)}")
    print(f"Total unique target patches: {len(all_target_patch_indices)}")
    
    if len(all_ref_patch_indices) > 0:
        print(f"Reference indices sample: {all_ref_patch_indices[:10]}...")
    if len(all_target_patch_indices) > 0:
        print(f"Target indices sample: {all_target_patch_indices[:10]}...")
    
    # Visualize patch locations
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 1. Original image with patch grid
    axes[0].imshow(img_array)
    axes[0].set_title(f"Original Image with Patch Grid\n{len(artifacts)} artifacts", fontweight='bold')
    
    # Draw patch grid
    for i in range(0, H, patch_size):
        axes[0].axhline(y=i, color='white', linewidth=0.5, alpha=0.3)
    for j in range(0, W, patch_size):
        axes[0].axvline(x=j, color='white', linewidth=0.5, alpha=0.3)
    
    # 2. Combined patches visualization (reference + target)
    axes[1].imshow(img_array)
    axes[1].set_title(f"All Patches Combined\nReference: {len(all_ref_patch_indices)} (Red), Target: {len(all_target_patch_indices)} (Blue)", fontweight='bold')
    
    # Overlay all reference patches (red)
    for idx in all_ref_patch_indices:
        # Subtract text length offset (512) to get visual patch index
        idx_visual = idx - 512
        if 0 <= idx_visual < patch_H * patch_W:
            row = idx_visual // patch_W
            col = idx_visual % patch_W
            y_start = row * patch_size
            x_start = col * patch_size
            rect = patches.Rectangle((x_start, y_start), patch_size, patch_size, 
                                    linewidth=2, edgecolor='red', facecolor='red', alpha=0.4)
            axes[1].add_patch(rect)
    
    # Overlay all target patches (blue)
    for idx in all_target_patch_indices:
        # Subtract text length offset (512) to get visual patch index
        idx_visual = idx - 512
        if 0 <= idx_visual < patch_H * patch_W:
            row = idx_visual // patch_W
            col = idx_visual % patch_W
            y_start = row * patch_size
            x_start = col * patch_size
            rect = patches.Rectangle((x_start, y_start), patch_size, patch_size, 
                                    linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.4)
            axes[1].add_patch(rect)
    
    for ax in axes:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Show bounding boxes and additional info for all artifacts
    print("\n=== DETAILED ARTIFACT INFO ===")
    for i, artifact in enumerate(artifacts):
        artifact_type = artifact.get('artifact_type', 'unknown')
        entity = artifact.get('entity', 'N/A')
        print(f"Artifact #{i+1} ({artifact_type} - {entity}):")
        
        if 'target_bbox' in artifact:
            bbox = artifact['target_bbox']
            print(f"  Target BBox: {bbox}")
        if 'reference_bbox' in artifact:
            bbox = artifact['reference_bbox']
            print(f"  Reference BBox: {bbox}")
        
        # Print distortion kernel for distortion artifacts
        if artifact_type == 'distortion' and 'distortion_kernel' in artifact:
            kernel = artifact['distortion_kernel']
            print(f"  Distortion Kernel: {kernel}")
        
        if 'target_bbox' not in artifact and 'reference_bbox' not in artifact:
            print("  No bounding boxes found")

In [5]:
from pipeline.flux_generator import FluxGenerator, FluxConfig

# Optionally, customize config
config = FluxConfig(
    name='flux-dev',
    guidance=5.0,
    num_steps=25,
    pe_step={'addition': 20, 'removal': 20, 'distortion': 20, 'fusion': 20},
    inject_step=15,
    seed=42,
    use_rf_solver=False,
    offload=False
)

flux_gen = FluxGenerator(device='cuda', config=config)

/home/jhpark/anaconda3/envs/rf-solver/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading FLUX models...


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Init model
Loading checkpoint
Init AE
FLUX models loaded successfully.


In [6]:
# Create an iterator for image directories to easily switch between them
removal_dir = Path('/data3/jhpark/person_1k')  # Current directory (exps/1k/removal)
image_dirs = [d for d in removal_dir.iterdir() if d.is_dir() and not d.name.startswith('log')]
image_dirs_iter = iter(image_dirs)

In [ ]:
exp_dir = '/data3/jhpark/fireflow/addition'
os.makedirs(exp_dir, exist_ok=True)
num_exps = 0
for dir_idx, selected_dir in enumerate(image_dirs):
    print(f"Loading data from: {selected_dir}")
    print("=" * 60)
    try:
        metadata = load_metadata(selected_dir)
    except Exception as e:
        print(f"Error loading metadata: {e}")
        continue
    caption = metadata['caption']

    if not any(artifact.get('artifact_type') == 'removal' for artifact in metadata.get('artifacts', [])):
        continue

    result_img = flux_gen.inject_artifacts(
        source_prompt=caption,
        target_prompt=caption,
        source_img=metadata['real_image_path'],
        artifact_data=metadata['artifacts'],
        pe_step_addition=20,
        pe_step_removal=24,
        pe_step_distortion=20,
        pe_step_fusion=20,
        inject_step=15,
    )

    # Create 1x3 grid plot (original image + patch mapping + result image)
    fig, axes = plt.subplots(1, 3, figsize=(36, 12))

    # Load original image for display
    orig_img = Image.open(metadata['real_image_path'])
    orig_img_array = np.array(orig_img)

    # Plot 1: Original image
    axes[0].imshow(orig_img_array)
    axes[0].set_title("Original Image", fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Plot 2: Original image with patch assignments
    axes[1].imshow(orig_img_array)
    axes[1].set_title("Patch Assignments\nReference (Red) + Target (Blue)", fontsize=14, fontweight='bold')

    # Get patch info from artifacts
    patch_size = 16
    H, W = orig_img_array.shape[:2]
    patch_H = H // patch_size
    patch_W = W // patch_size

    # Collect all patch indices from all artifacts for visualization
    all_ref_patch_indices = []
    all_target_patch_indices = []

    for artifact in metadata['artifacts']:
        ref_indices = artifact.get('reference_patch_indices', []) or []
        target_indices = artifact.get('target_patch_indices', []) or []
        all_ref_patch_indices.extend(ref_indices)
        all_target_patch_indices.extend(target_indices)

    # Remove duplicates while preserving order
    all_ref_patch_indices = list(dict.fromkeys(all_ref_patch_indices))
    all_target_patch_indices = list(dict.fromkeys(all_target_patch_indices))

    # Overlay all reference patches (red)
    for idx in all_ref_patch_indices:
        idx_visual = idx - 512  # Subtract text length offset
        if 0 <= idx_visual < patch_H * patch_W:
            row = idx_visual // patch_W
            col = idx_visual % patch_W
            y_start = row * patch_size
            x_start = col * patch_size
            rect = patches.Rectangle((x_start, y_start), patch_size, patch_size, 
                                    linewidth=2, edgecolor='red', facecolor='red', alpha=0.4)
            axes[1].add_patch(rect)

    # Overlay all target patches (blue)
    for idx in all_target_patch_indices:
        idx_visual = idx - 512  # Subtract text length offset
        if 0 <= idx_visual < patch_H * patch_W:
            row = idx_visual // patch_W
            col = idx_visual % patch_W
            y_start = row * patch_size
            x_start = col * patch_size
            rect = patches.Rectangle((x_start, y_start), patch_size, patch_size, 
                                    linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.4)
            axes[1].add_patch(rect)

    # Create bounding boxes around target patches for each artifact
    # Color mapping by artifact type (same color for same type)
    type_colors = {
        'removal': 'red',
        'addition': 'green', 
        'distortion': 'orange',
        'fusion': 'purple',
        'unknown': 'gray'
    }

    for artifact_idx, artifact in enumerate(metadata['artifacts']):
        artifact_type = artifact.get('artifact_type', 'unknown')
        target_indices = artifact.get('target_patch_indices', []) or []
        
        # Determine what text to display
        if artifact_type == 'distortion' and 'distortion_kernel' in artifact:
            display_text = str(artifact['distortion_kernel'])
        else:
            display_text = artifact_type
        
        if target_indices:
            # Find bounding box coordinates from target patches
            min_x, max_x = float('inf'), -1
            min_y, max_y = float('inf'), -1
            
            for idx in target_indices:
                idx_visual = idx - 512  # Subtract text length offset
                if 0 <= idx_visual < patch_H * patch_W:
                    row = idx_visual // patch_W
                    col = idx_visual % patch_W
                    y_start = row * patch_size
                    x_start = col * patch_size
                    
                    min_x = min(min_x, x_start)
                    max_x = max(max_x, x_start + patch_size)
                    min_y = min(min_y, y_start)
                    max_y = max(max_y, y_start + patch_size)
            
            # Draw bounding box around target patches
            if min_x != float('inf'):
                bbox_color = type_colors.get(artifact_type, 'gray')
                width = max_x - min_x
                height = max_y - min_y
                
                # Draw the bounding box
                bbox_rect = patches.Rectangle((min_x, min_y), width, height, 
                                            linewidth=3, edgecolor=bbox_color, 
                                            facecolor='none', linestyle='--')
                axes[1].add_patch(bbox_rect)
                
                # Add label at top-left of bounding box
                axes[1].text(min_x, min_y - 5, display_text, 
                            fontsize=18, fontweight='bold', color='black',
                            ha='left', va='bottom',
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', 
                                    edgecolor=bbox_color, alpha=0.9))

    axes[1].axis('off')
    
    # Plot 3: Generated result image
    axes[2].imshow(result_img)
    axes[2].set_title("Generated Result", fontsize=14, fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    fig.savefig(os.path.join(exp_dir, f"{dir_idx}.png"))
    plt.close()

Loading data from: /data3/jhpark/person_1k/8ac2a665-0e3d-49e0-93d0-7a0f32ca1397
Error loading metadata: metadata.pkl not found in /data3/jhpark/person_1k/8ac2a665-0e3d-49e0-93d0-7a0f32ca1397
Loading data from: /data3/jhpark/person_1k/cde0d8e0-f302-42ed-a01c-abfc350515fd
Loading data from: /data3/jhpark/person_1k/b45e6f70-8b42-41b3-a422-253e89a21023
Loading data from: /data3/jhpark/person_1k/b746f636-9876-47c0-9414-8ac2467befaf
Error loading metadata: metadata.pkl not found in /data3/jhpark/person_1k/b746f636-9876-47c0-9414-8ac2467befaf
Loading data from: /data3/jhpark/person_1k/ead32f5d-4d10-46dc-8e88-c29bdfc00a66
Loading data from: /data3/jhpark/person_1k/2baad4eb-4f3e-477c-8a85-ecf560bce55e
Loading data from: /data3/jhpark/person_1k/681076b2-753f-4645-8452-041a098fadef
Loading data from: /data3/jhpark/person_1k/22173dbe-054b-4b67-a71e-7d5d730cf6d5
Loading data from: /data3/jhpark/person_1k/7298bd19-cf67-4b70-9d3e-9de98d8201c0
Loading data from: /data3/jhpark/person_1k/25d5ce31-bfc5-4

RuntimeError: shape mismatch: value tensor of shape [16, 64, 2, 2] cannot be broadcast to indexing result of shape [1, 1, 7, 64, 2, 2]

In [19]:
exp_dir = '/data3/jhpark/fireflow6'
os.makedirs(exp_dir, exist_ok=True)
for dir_idx, selected_dir in enumerate(image_dirs[:20]):
    print(f"Loading data from: {selected_dir}")
    print("=" * 60)
    try:
        metadata = load_metadata(selected_dir)
    except Exception as e:
        print(f"Error loading metadata: {e}")
        continue
    caption = metadata['caption']


    result_img = flux_gen.inject_artifacts(
        source_prompt=caption,
        target_prompt=caption,
        source_img=metadata['real_image_path'],
        artifact_data=metadata['artifacts'],
        pe_step_addition=9,
        pe_step_removal=9,
        pe_step_distortion=6,
        pe_step_fusion=6,
        inject_step=6,
        num_steps=10,
    )

    # Create 1x3 grid plot (original image + patch mapping + result image)
    fig, axes = plt.subplots(1, 3, figsize=(36, 12))

    # Load original image for display
    orig_img = Image.open(metadata['real_image_path'])
    orig_img_array = np.array(orig_img)

    # Plot 1: Original image
    axes[0].imshow(orig_img_array)
    axes[0].set_title("Original Image", fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Plot 2: Original image with patch assignments
    axes[1].imshow(orig_img_array)
    axes[1].set_title("Patch Assignments\nReference (Red) + Target (Blue)", fontsize=14, fontweight='bold')

    # Get patch info from artifacts
    patch_size = 16
    H, W = orig_img_array.shape[:2]
    patch_H = H // patch_size
    patch_W = W // patch_size

    # Collect all patch indices from all artifacts for visualization
    all_ref_patch_indices = []
    all_target_patch_indices = []

    for artifact in metadata['artifacts']:
        ref_indices = artifact.get('reference_patch_indices', []) or []
        target_indices = artifact.get('target_patch_indices', []) or []
        all_ref_patch_indices.extend(ref_indices)
        all_target_patch_indices.extend(target_indices)

    # Remove duplicates while preserving order
    all_ref_patch_indices = list(dict.fromkeys(all_ref_patch_indices))
    all_target_patch_indices = list(dict.fromkeys(all_target_patch_indices))

    # Overlay all reference patches (red)
    for idx in all_ref_patch_indices:
        idx_visual = idx - 512  # Subtract text length offset
        if 0 <= idx_visual < patch_H * patch_W:
            row = idx_visual // patch_W
            col = idx_visual % patch_W
            y_start = row * patch_size
            x_start = col * patch_size
            rect = patches.Rectangle((x_start, y_start), patch_size, patch_size, 
                                    linewidth=2, edgecolor='red', facecolor='red', alpha=0.4)
            axes[1].add_patch(rect)

    # Overlay all target patches (blue)
    for idx in all_target_patch_indices:
        idx_visual = idx - 512  # Subtract text length offset
        if 0 <= idx_visual < patch_H * patch_W:
            row = idx_visual // patch_W
            col = idx_visual % patch_W
            y_start = row * patch_size
            x_start = col * patch_size
            rect = patches.Rectangle((x_start, y_start), patch_size, patch_size, 
                                    linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.4)
            axes[1].add_patch(rect)

    # Create bounding boxes around target patches for each artifact
    # Color mapping by artifact type (same color for same type)
    type_colors = {
        'removal': 'red',
        'addition': 'green', 
        'distortion': 'orange',
        'fusion': 'purple',
        'unknown': 'gray'
    }

    for artifact_idx, artifact in enumerate(metadata['artifacts']):
        artifact_type = artifact.get('artifact_type', 'unknown')
        target_indices = artifact.get('target_patch_indices', []) or []
        
        # Determine what text to display
        if artifact_type == 'distortion' and 'distortion_kernel' in artifact:
            display_text = str(artifact['distortion_kernel'])
        else:
            display_text = artifact_type
        
        if target_indices:
            # Find bounding box coordinates from target patches
            min_x, max_x = float('inf'), -1
            min_y, max_y = float('inf'), -1
            
            for idx in target_indices:
                idx_visual = idx - 512  # Subtract text length offset
                if 0 <= idx_visual < patch_H * patch_W:
                    row = idx_visual // patch_W
                    col = idx_visual % patch_W
                    y_start = row * patch_size
                    x_start = col * patch_size
                    
                    min_x = min(min_x, x_start)
                    max_x = max(max_x, x_start + patch_size)
                    min_y = min(min_y, y_start)
                    max_y = max(max_y, y_start + patch_size)
            
            # Draw bounding box around target patches
            if min_x != float('inf'):
                bbox_color = type_colors.get(artifact_type, 'gray')
                width = max_x - min_x
                height = max_y - min_y
                
                # Draw the bounding box
                bbox_rect = patches.Rectangle((min_x, min_y), width, height, 
                                            linewidth=3, edgecolor=bbox_color, 
                                            facecolor='none', linestyle='--')
                axes[1].add_patch(bbox_rect)
                
                # Add label at top-left of bounding box
                axes[1].text(min_x, min_y - 5, display_text, 
                            fontsize=18, fontweight='bold', color='black',
                            ha='left', va='bottom',
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', 
                                    edgecolor=bbox_color, alpha=0.9))

    axes[1].axis('off')
    
    # Plot 3: Generated result image
    axes[2].imshow(result_img)
    axes[2].set_title("Generated Result", fontsize=14, fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    fig.savefig(os.path.join(exp_dir, f"{dir_idx}.png"))
    plt.close()

Loading data from: /data3/jhpark/person_1k/8ac2a665-0e3d-49e0-93d0-7a0f32ca1397
Error loading metadata: metadata.pkl not found in /data3/jhpark/person_1k/8ac2a665-0e3d-49e0-93d0-7a0f32ca1397
Loading data from: /data3/jhpark/person_1k/cde0d8e0-f302-42ed-a01c-abfc350515fd
Generating with seed 42:
a man is standing next to an elephant
Done in 6.2s.
Loading data from: /data3/jhpark/person_1k/b45e6f70-8b42-41b3-a422-253e89a21023
Generating with seed 42:
a group of buses parked in a row
Done in 6.0s.
Loading data from: /data3/jhpark/person_1k/b746f636-9876-47c0-9414-8ac2467befaf
Error loading metadata: metadata.pkl not found in /data3/jhpark/person_1k/b746f636-9876-47c0-9414-8ac2467befaf
Loading data from: /data3/jhpark/person_1k/ead32f5d-4d10-46dc-8e88-c29bdfc00a66
Generating with seed 42:
a woman eating a hot dog
Done in 5.7s.
Loading data from: /data3/jhpark/person_1k/2baad4eb-4f3e-477c-8a85-ecf560bce55e
Generating with seed 42:
a man riding a skateboard
Done in 9.6s.
Loading data from: /

In [ ]:
exp_dir = '/data3/jhpark/testing11'
os.makedirs(exp_dir, exist_ok=True)
for dir_idx, selected_dir in enumerate(image_dirs[:100]):
    print(f"Loading data from: {selected_dir}")
    print("=" * 60)
    try:
        metadata = load_metadata(selected_dir)
    except Exception as e:
        print(f"Error loading metadata: {e}")
        continue
    caption = metadata['caption']


    result_img = flux_gen.inject_artifacts(
        source_prompt=caption,
        target_prompt=caption,
        source_img=metadata['real_image_path'],
        artifact_data=metadata['artifacts'],
        pe_step_addition=25,
        pe_step_removal=25,
        pe_step_distortion=20,
        pe_step_fusion=20,
        inject_step=20,
    )

    # Create 1x3 grid plot (original image + patch mapping + result image)
    fig, axes = plt.subplots(1, 3, figsize=(36, 12))

    # Load original image for display
    orig_img = Image.open(metadata['real_image_path'])
    orig_img_array = np.array(orig_img)

    # Plot 1: Original image
    axes[0].imshow(orig_img_array)
    axes[0].set_title("Original Image", fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Plot 2: Original image with patch assignments
    axes[1].imshow(orig_img_array)
    axes[1].set_title("Patch Assignments\nReference (Red) + Target (Blue)", fontsize=14, fontweight='bold')

    # Get patch info from artifacts
    patch_size = 16
    H, W = orig_img_array.shape[:2]
    patch_H = H // patch_size
    patch_W = W // patch_size

    # Collect all patch indices from all artifacts for visualization
    all_ref_patch_indices = []
    all_target_patch_indices = []

    for artifact in metadata['artifacts']:
        ref_indices = artifact.get('reference_patch_indices', []) or []
        target_indices = artifact.get('target_patch_indices', []) or []
        all_ref_patch_indices.extend(ref_indices)
        all_target_patch_indices.extend(target_indices)

    # Remove duplicates while preserving order
    all_ref_patch_indices = list(dict.fromkeys(all_ref_patch_indices))
    all_target_patch_indices = list(dict.fromkeys(all_target_patch_indices))

    # Overlay all reference patches (red)
    for idx in all_ref_patch_indices:
        idx_visual = idx - 512  # Subtract text length offset
        if 0 <= idx_visual < patch_H * patch_W:
            row = idx_visual // patch_W
            col = idx_visual % patch_W
            y_start = row * patch_size
            x_start = col * patch_size
            rect = patches.Rectangle((x_start, y_start), patch_size, patch_size, 
                                    linewidth=2, edgecolor='red', facecolor='red', alpha=0.4)
            axes[1].add_patch(rect)

    # Overlay all target patches (blue)
    for idx in all_target_patch_indices:
        idx_visual = idx - 512  # Subtract text length offset
        if 0 <= idx_visual < patch_H * patch_W:
            row = idx_visual // patch_W
            col = idx_visual % patch_W
            y_start = row * patch_size
            x_start = col * patch_size
            rect = patches.Rectangle((x_start, y_start), patch_size, patch_size, 
                                    linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.4)
            axes[1].add_patch(rect)

    # Create bounding boxes around target patches for each artifact
    # Color mapping by artifact type (same color for same type)
    type_colors = {
        'removal': 'red',
        'addition': 'green', 
        'distortion': 'orange',
        'fusion': 'purple',
        'unknown': 'gray'
    }

    for artifact_idx, artifact in enumerate(metadata['artifacts']):
        artifact_type = artifact.get('artifact_type', 'unknown')
        target_indices = artifact.get('target_patch_indices', []) or []
        
        # Determine what text to display
        if artifact_type == 'distortion' and 'distortion_kernel' in artifact:
            display_text = str(artifact['distortion_kernel'])
        else:
            display_text = artifact_type
        
        if target_indices:
            # Find bounding box coordinates from target patches
            min_x, max_x = float('inf'), -1
            min_y, max_y = float('inf'), -1
            
            for idx in target_indices:
                idx_visual = idx - 512  # Subtract text length offset
                if 0 <= idx_visual < patch_H * patch_W:
                    row = idx_visual // patch_W
                    col = idx_visual % patch_W
                    y_start = row * patch_size
                    x_start = col * patch_size
                    
                    min_x = min(min_x, x_start)
                    max_x = max(max_x, x_start + patch_size)
                    min_y = min(min_y, y_start)
                    max_y = max(max_y, y_start + patch_size)
            
            # Draw bounding box around target patches
            if min_x != float('inf'):
                bbox_color = type_colors.get(artifact_type, 'gray')
                width = max_x - min_x
                height = max_y - min_y
                
                # Draw the bounding box
                bbox_rect = patches.Rectangle((min_x, min_y), width, height, 
                                            linewidth=3, edgecolor=bbox_color, 
                                            facecolor='none', linestyle='--')
                axes[1].add_patch(bbox_rect)
                
                # Add label at top-left of bounding box
                axes[1].text(min_x, min_y - 5, display_text, 
                            fontsize=18, fontweight='bold', color='black',
                            ha='left', va='bottom',
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', 
                                    edgecolor=bbox_color, alpha=0.9))

    axes[1].axis('off')
    
    # Plot 3: Generated result image
    axes[2].imshow(result_img)
    axes[2].set_title("Generated Result", fontsize=14, fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    fig.savefig(os.path.join(exp_dir, f"{dir_idx}.png"))
    plt.close()

Loading data from: /data3/jhpark/person_1k/8ac2a665-0e3d-49e0-93d0-7a0f32ca1397
Error loading metadata: metadata.pkl not found in /data3/jhpark/person_1k/8ac2a665-0e3d-49e0-93d0-7a0f32ca1397
Loading data from: /data3/jhpark/person_1k/cde0d8e0-f302-42ed-a01c-abfc350515fd
Generating with seed 42:
a man is standing next to an elephant
Done in 14.9s.
Loading data from: /data3/jhpark/person_1k/b45e6f70-8b42-41b3-a422-253e89a21023
Generating with seed 42:
a group of buses parked in a row
Done in 14.1s.
Loading data from: /data3/jhpark/person_1k/b746f636-9876-47c0-9414-8ac2467befaf
Error loading metadata: metadata.pkl not found in /data3/jhpark/person_1k/b746f636-9876-47c0-9414-8ac2467befaf
Loading data from: /data3/jhpark/person_1k/ead32f5d-4d10-46dc-8e88-c29bdfc00a66
Generating with seed 42:
a woman eating a hot dog
Done in 13.4s.
Loading data from: /data3/jhpark/person_1k/2baad4eb-4f3e-477c-8a85-ecf560bce55e
Generating with seed 42:
a man riding a skateboard
Done in 22.6s.
Loading data fro

: 